In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Mode: 'sector' or 'tenor' ---
MODE = 'sector'  # 'sector' or 'tenor'

SECTORS = {
    'Belly': ['6Y', '7Y', '8Y'],
    '10Y Sector': ['9Y', '10Y', '11Y'],
    'Long End': ['12Y', '13Y', '14Y', '15Y'],
}

SELECTED = 'Belly'  # sector name if MODE='sector', or tenor string like '10Y' if MODE='tenor'

COUNTRIES = {
    'Peru': df_perugb_cmt,
    'Mexico': df_mbono_cmt,
    'Colombia': df_coltes_cmt,
    'Chile': df_btpcl_cmt,
}

FOCUS_COUNTRY = 'Peru'
LOOKBACK_START = '2022-01-01'  # controls everything — no data before this date used anywhere
N_PCS = 2  # None = auto via 90% cumvar

SD_WINDOW = 252
SD_BANDS = [1.25, 1.65]
ROLLING_WINDOWS = [5, 10, 20]
ROLL_DISPLAY = [5, 20]
TRAIL_WINDOW = 60
ATTRIB_WINDOWS = [5, 10, 20]

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.15,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 120,
})

# institutional palette
COLORS = {
    'pc1': '#4A6FA5',      # steel blue
    'pc2': '#C47B3B',      # copper
    'pc3': '#6B8F6B',      # sage
    'mean': '#8B8B8B',     # warm gray
    'actual': '#1A1A1A',   # near black
    'pos_inner': '#FFDAB9', # light peach
    'pos_outer': '#CD5C5C', # indian red
    'neg_inner': '#B0D4F1', # light steel blue
    'neg_outer': '#2E5E8E', # dark steel blue
    'bull': '#C8E6C9',     # light green
    'bear': '#FFCDD2',     # light red
}

In [ ]:
def build_spread_panel(countries, mode, selected, sectors, ust_df, lookback_start):
    # determine UST series
    ust = ust_df.set_index('Fecha') if 'Fecha' in ust_df.columns else ust_df.copy()
    if mode == 'sector':
        cols = sectors[selected]
        ust_series = ust[cols].mean(axis=1)
    else:
        ust_series = ust[selected]
    result = {}
    for name, df in countries.items():
        cty = df.set_index('Fecha') if 'Fecha' in df.columns else df.copy()
        if mode == 'sector':
            cty_series = cty[cols].mean(axis=1)
        else:
            cty_series = cty[selected]
        # intersect dates
        idx = cty_series.index.intersection(ust_series.index)
        spread = (cty_series.loc[idx] - ust_series.loc[idx]) * 100
        result[name] = spread
    panel = pd.DataFrame(result)
    panel.index = pd.to_datetime(panel.index)
    # filter to lookback_start immediately
    panel = panel.loc[panel.index >= pd.Timestamp(lookback_start)]
    return panel

df_spreads = build_spread_panel(COUNTRIES, MODE, SELECTED, SECTORS, df_ust_cmt, LOOKBACK_START)
print(f'Mode: {MODE} | Selected: {SELECTED}')
print(f'Shape: {df_spreads.shape} | {df_spreads.index[0].date()} to {df_spreads.index[-1].date()}')
print(df_spreads.head(3))

In [ ]:
def run_pca(df_spreads, n_pcs):
    data = df_spreads.dropna()
    means = data.mean()
    demeaned = data - means
    cov = demeaned.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # sort descending
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    total_var = eigenvalues.sum()
    var_explained = eigenvalues / total_var
    cum_var = np.cumsum(var_explained)
    # determine n_pcs
    if n_pcs is None:
        k = int(np.searchsorted(cum_var, 0.90)) + 1
    else:
        k = n_pcs
    loadings = pd.DataFrame(
        eigenvectors[:, :k],
        index=data.columns,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    scores = pd.DataFrame(
        demeaned.values @ eigenvectors[:, :k],
        index=data.index,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    return {
        'means': means,
        'loadings': loadings,
        'scores': scores,
        'eigenvalues': eigenvalues[:k],
        'var_explained': var_explained[:k],
        'cum_var_explained': cum_var[:k],
        'n_pcs': k,
    }

def decompose(pca, df_spreads):
    data = df_spreads.dropna()
    loadings = pca['loadings']
    means = pca['means']
    demeaned = data - means
    scores_full = pd.DataFrame(
        demeaned.values @ loadings.values,
        index=data.index,
        columns=loadings.columns
    )
    fitted = pd.DataFrame(
        scores_full.values @ loadings.values.T + means.values,
        index=data.index,
        columns=data.columns
    )
    residuals = data - fitted
    # per-factor contributions per country: score_k * loading_country_k
    contributions = {}
    for country in data.columns:
        contribs = pd.DataFrame(
            {pc: scores_full[pc] * loadings.loc[country, pc] for pc in loadings.columns},
            index=data.index
        )
        contributions[country] = contribs
    return {
        'fitted': fitted,
        'residuals': residuals,
        'contributions': contributions,
        'scores_full': scores_full,
    }

pca = run_pca(df_spreads, N_PCS)
decomp = decompose(pca, df_spreads)
print(f'n_pcs: {pca["n_pcs"]}')
print(f'Var explained: {[f"{v:.1%}" for v in pca["var_explained"]]}')
print(f'Cumulative: {[f"{v:.1%}" for v in pca["cum_var_explained"]]}')
print(f'Score date range: {decomp["scores_full"].index[0].date()} to {decomp["scores_full"].index[-1].date()}')
print(f'Residual shape: {decomp["residuals"].shape}')

In [ ]:
# daily factor returns (first difference of scores)
factor_returns = decomp['scores_full'].diff().dropna()

# rolling sums for each window
roll_factor = {}
for w in ROLLING_WINDOWS:
    roll_factor[w] = factor_returns.rolling(w).sum()

print(f'Factor returns shape: {factor_returns.shape}')
print(f'Rolling windows computed: {list(roll_factor.keys())}')
print(factor_returns.describe().round(3))

In [ ]:
# --- Variance Explained Table ---
ve_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(pca['n_pcs'])],
    'Var Explained (%)': [f'{v:.2%}' for v in pca['var_explained']],
    'Cumulative (%)': [f'{v:.2%}' for v in pca['cum_var_explained']],
})
print(ve_df.to_string(index=False))

pc_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
n = pca['n_pcs']

# --- Bar chart 1: Variance Explained per PC ---
fig, ax = plt.subplots(figsize=(max(4, n * 1.2), 4))
bars = ax.bar(pcs, pca['var_explained'] * 100, color=pc_colors[:n], width=0.5, edgecolor='white')
for bar, v in zip(bars, pca['var_explained']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{v:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Variance Explained per PC ({MODE.title()}: {SELECTED})')
ax.set_ylabel('Variance Explained (%)')
ax.set_ylim(0, max(pca['var_explained']) * 100 * 1.2)
plt.tight_layout()
plt.show()

# --- Bar chart 2: Grouped Loadings by Country ---
loadings_df = pca['loadings']
countries_list = list(loadings_df.index)
n_countries = len(countries_list)
x = np.arange(n_countries)
width = 0.8 / n

fig, ax = plt.subplots(figsize=(max(6, n_countries * 1.5), 4))
for i, pc in enumerate(pcs):
    offset = (i - (n - 1) / 2) * width
    vals = loadings_df[pc].values
    bars = ax.bar(x + offset, vals, width=width * 0.9,
                  label=pc, color=pc_colors[i], edgecolor='white')
    for bar, v in zip(bars, vals):
        va = 'bottom' if v >= 0 else 'top'
        y_off = 0.005 if v >= 0 else -0.005
        ax.text(bar.get_x() + bar.get_width() / 2, v + y_off,
                f'{v:.2f}', ha='center', va=va, fontsize=7)
ax.axhline(0, color='#555555', linewidth=0.8, linestyle='-')
ax.set_xticks(x)
ax.set_xticklabels(countries_list)
ax.set_title(f'PCA Loadings by Country ({"Sector" if MODE == "sector" else "Tenor"}: {SELECTED})')
ax.set_ylabel('Loading')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Part 1: Loading vs Volatility ---
spread_changes = df_spreads.dropna().diff().dropna()
sd_per_country = spread_changes.std()

# SD ratios relative to first country
base = sd_per_country.iloc[0]
sd_ratios = sd_per_country / base

pc1_loadings = pca['loadings']['PC1'].abs()
base_load = pc1_loadings.iloc[0]
load_ratios = pc1_loadings / base_load

ratio_df = pd.DataFrame({
    'SD (bps)': sd_per_country.round(2),
    'SD Ratio': sd_ratios.round(3),
    '|PC1 Loading|': pc1_loadings.round(4),
    'Loading Ratio': load_ratios.round(3),
    'Ratio Match': (sd_ratios / load_ratios).round(3),
})
print('Loading vs Volatility (base = first country):')
print(ratio_df.to_string())

# --- Part 2: PCA on Correlation Matrix ---
def run_pca_corr(df_spreads, n_pcs):
    data = df_spreads.dropna()
    stds = data.std()
    standardized = data / stds
    corr = standardized.cov().values
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    total_var = eigenvalues.sum()
    var_explained = eigenvalues / total_var
    cum_var = np.cumsum(var_explained)
    if n_pcs is None:
        k = int(np.searchsorted(cum_var, 0.90)) + 1
    else:
        k = n_pcs
    loadings = pd.DataFrame(
        eigenvectors[:, :k],
        index=data.columns,
        columns=[f'PC{i+1}' for i in range(k)]
    )
    return {
        'loadings': loadings,
        'var_explained': var_explained[:k],
        'cum_var_explained': cum_var[:k],
        'n_pcs': k,
    }

pca_corr = run_pca_corr(df_spreads, N_PCS)

# print side-by-side
cov_loads = pca['loadings'].copy()
corr_loads = pca_corr['loadings'].copy()
cov_loads.columns = [f'Cov_{c}' for c in cov_loads.columns]
corr_loads.columns = [f'Corr_{c}' for c in corr_loads.columns]
combined = pd.concat([cov_loads, corr_loads], axis=1)
print('\nCovariance vs Correlation Loadings:')
print(combined.round(4).to_string())

# side-by-side bar charts
pcs = [f'PC{i+1}' for i in range(pca['n_pcs'])]
n = len(pcs)
pc_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]
countries_list = list(pca['loadings'].index)
n_countries = len(countries_list)
x = np.arange(n_countries)
width = 0.8 / n

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Covariance vs Correlation Loadings ({MODE.title()}: {SELECTED})',
             fontsize=12, fontweight='bold')

for ax_i, (ax, lds, label) in enumerate([
    (axes[0], pca['loadings'], 'Covariance PCA'),
    (axes[1], pca_corr['loadings'], 'Correlation PCA'),
]):
    for i, pc in enumerate(pcs):
        offset = (i - (n - 1) / 2) * width
        vals = lds[pc].values
        bars = ax.bar(x + offset, vals, width=width * 0.9,
                      label=pc, color=pc_colors[i], edgecolor='white')
        for bar, v in zip(bars, vals):
            va = 'bottom' if v >= 0 else 'top'
            y_off = 0.005 if v >= 0 else -0.005
            ax.text(bar.get_x() + bar.get_width() / 2, v + y_off,
                    f'{v:.2f}', ha='center', va=va, fontsize=7)
    ax.axhline(0, color='#555555', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(countries_list)
    ax.set_title(label)
    ax.set_ylabel('Loading')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 7: PC Score Levels with Regime Shading ---
scores = decomp['scores_full']
n_pcs = pca['n_pcs']
pc_line_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['pc3']]

fig, axes = plt.subplots(n_pcs, 1, figsize=(12, 3 * n_pcs), sharex=True)
if n_pcs == 1:
    axes = [axes]

for i, pc in enumerate([f'PC{k+1}' for k in range(n_pcs)]):
    ax = axes[i]
    s = scores[pc]
    trail = s.rolling(TRAIL_WINDOW).mean()
    color = pc_line_colors[i]
    # regime shading
    ax.fill_between(s.index, s.min() * 1.5, s.max() * 1.5,
                    where=(s >= trail), color=COLORS['bull'], alpha=0.5, label='_nolegend_')
    ax.fill_between(s.index, s.min() * 1.5, s.max() * 1.5,
                    where=(s < trail), color=COLORS['bear'], alpha=0.5, label='_nolegend_')
    ax.plot(s.index, s, color=color, linewidth=1.2, label=f'{pc} level')
    ax.plot(trail.index, trail, color=color, linewidth=1.0, linestyle='--',
            alpha=0.55, label=f'{TRAIL_WINDOW}d mean')
    ax.axhline(0, color='#999999', linewidth=0.7)
    ax.set_ylabel('Score (bps)')
    ax.legend(loc='upper left')
    ax.set_xlim(scores.index[0], scores.index[-1])
    # keep y-axis tight to actual data range
    pad = (s.max() - s.min()) * 0.05
    ax.set_ylim(s.min() - pad, s.max() + pad)
    ax.set_title(pc)

fig.suptitle(f'PC Scores — Levels & Regime ({MODE}: {SELECTED})', y=1.01)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 8: Factor Return Momentum ---
roll_colors = [COLORS['pc1'], COLORS['pc2'], COLORS['mean'], COLORS['pc3']]
n_pcs = pca['n_pcs']

fig, axes = plt.subplots(n_pcs, 1, figsize=(12, 3 * n_pcs), sharex=True)
if n_pcs == 1:
    axes = [axes]

for i, pc in enumerate([f'PC{k+1}' for k in range(n_pcs)]):
    ax = axes[i]
    for j, w in enumerate(ROLL_DISPLAY):
        series = roll_factor[w][pc].dropna()
        ax.plot(series.index, series, color=roll_colors[j], linewidth=1.2, label=f'{w}d sum')
    ax.axhline(0, color='#999999', linewidth=0.7)
    ax.set_ylabel('Rolling Sum (bps)')
    ax.legend(loc='upper left')
    ax.set_xlim(factor_returns.index[0], factor_returns.index[-1])
    ax.set_title(pc)

fig.suptitle(f'Factor Return Momentum ({MODE}: {SELECTED})', y=1.01)
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# --- Cell 9: Regime Summary Table (Client-Facing) ---
def regime_summary(scores_full, roll_factor, pca, trail_window):
    latest = scores_full.index[-1]
    rows = []
    for i in range(pca['n_pcs']):
        pc = f'PC{i+1}'
        s = scores_full[pc]
        level = float(s.iloc[-1])
        pctile = float((s < level).mean() * 100)
        mom5 = float(roll_factor[5][pc].iloc[-1])
        mom20 = float(roll_factor[20][pc].iloc[-1])
        trail_mean = float(s.rolling(trail_window).mean().iloc[-1])
        regime = 'Above trend' if level > trail_mean else 'Below trend'
        # direction: PC1 positive score = spreads wide (loadings typically positive)
        # use 20d momentum sign to determine direction
        if pc == 'PC1':
            direction = 'Compressing' if mom20 < 0 else 'Widening'
        else:
            direction = 'Differentiating'
        rows.append({
            'Factor': pc,
            'Level': level,
            '%ile': pctile,
            '5d Mom': mom5,
            '20d Mom': mom20,
            'Direction': direction,
            'Regime': regime,
        })
    # PC2 differentiation intensity
    pc2_level = float(scores_full['PC2'].iloc[-1]) if pca['n_pcs'] >= 2 else None
    pc2_pctile = float((scores_full['PC2'] < pc2_level).mean() * 100) if pc2_level is not None else None
    diff_intensity = abs(pc2_pctile - 50) if pc2_pctile is not None else None
    return rows, latest, diff_intensity, pc2_pctile

rows, latest, diff_intensity, pc2_pctile = regime_summary(
    decomp['scores_full'], roll_factor, pca, TRAIL_WINDOW
)

# format and print
header = f'=== LatAm LC Regime Monitor | {MODE}: {SELECTED} | {latest.date()} ==='
print(header)
print()
col_w = [8, 8, 6, 9, 9, 18, 12]
hdrs = ['Factor', 'Level', '%ile', '5d Mom', '20d Mom', 'Direction', 'Regime']
print(''.join(h.ljust(col_w[j]) for j, h in enumerate(hdrs)))
for r in rows:
    vals = [
        r['Factor'],
        f"{r['Level']:+.1f}",
        f"{r['%ile']:.0f}%",
        f"{r['5d Mom']:+.1f}",
        f"{r['20d Mom']:+.1f}",
        r['Direction'],
        r['Regime'],
    ]
    print(''.join(str(v).ljust(col_w[j]) for j, v in enumerate(vals)))

if diff_intensity is not None:
    if diff_intensity >= 35:
        intensity_label = 'High'
    elif diff_intensity >= 15:
        intensity_label = 'Moderate'
    else:
        intensity_label = 'Low'
    print(f'\nDifferentiation intensity: {intensity_label} (PC2 at {pc2_pctile:.0f}th percentile)')

In [ ]:
# --- Cell 10: Client Flow Anticipation ---
def classify_momentum_state(fast, slow, ratio=5/20):
    # returns string label for each date
    states = []
    for f, s in zip(fast, slow):
        if pd.isna(f) or pd.isna(s):
            states.append('unknown')
            continue
        same_sign = (f > 0 and s > 0) or (f < 0 and s < 0)
        building = abs(f) > abs(s) * ratio
        if f > 0 and s > 0 and building:
            states.append('accel_compress')
        elif f < 0 and s < 0 and building:
            states.append('accel_widen')
        elif not same_sign:
            states.append('transition')
        else:
            states.append('decel')
    return states

fast = roll_factor[5]['PC1']
slow = roll_factor[20]['PC1']
accel = fast.abs() - slow.abs() * (5 / 20)

states = classify_momentum_state(fast.values, slow.values)
state_series = pd.Series(states, index=fast.index)

state_color_map = {
    'accel_compress': COLORS['bull'],
    'accel_widen': COLORS['bear'],
    'decel': '#E0E0E0',
    'transition': '#E0E0E0',
    'unknown': '#FFFFFF',
}

# bar colors for acceleration panel
bar_colors = []
for st, av in zip(state_series, accel):
    if st == 'accel_compress':
        bar_colors.append(COLORS['pc3'])   # sage green
    elif st == 'accel_widen':
        bar_colors.append(COLORS['pos_outer'])  # indian red
    else:
        bar_colors.append(COLORS['mean'])   # gray

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
fig.suptitle(f'Client Flow Anticipation — PC1 ({MODE}: {SELECTED})', y=1.01)

# top: fast and slow rolling sums with regime shading
valid = fast.dropna().index
ax1.fill_between(state_series.index,
                 fast.min() * 1.5, fast.max() * 1.5,
                 where=(state_series == 'accel_compress'),
                 color=COLORS['bull'], alpha=0.4, label='_nolegend_')
ax1.fill_between(state_series.index,
                 fast.min() * 1.5, fast.max() * 1.5,
                 where=(state_series == 'accel_widen'),
                 color=COLORS['bear'], alpha=0.4, label='_nolegend_')
ax1.plot(fast.index, fast, color=COLORS['pc1'], linewidth=1.2, label='5d sum')
ax1.plot(slow.index, slow, color=COLORS['pc2'], linewidth=1.2,
         linestyle='--', label='20d sum')
ax1.axhline(0, color='#999999', linewidth=0.7)
ax1.set_ylabel('Rolling Sum (bps)')
ax1.legend(loc='upper left')
fmin, fmax = fast.min(), fast.max()
pad1 = (fmax - fmin) * 0.05
ax1.set_ylim(fmin - pad1, fmax + pad1)

# bottom: acceleration bars
accel_clean = accel.dropna()
state_clean = state_series.loc[accel_clean.index]
colors_clean = []
for st, av in zip(state_clean, accel_clean):
    if st == 'accel_compress':
        colors_clean.append(COLORS['pc3'])
    elif st == 'accel_widen':
        colors_clean.append(COLORS['pos_outer'])
    else:
        colors_clean.append(COLORS['mean'])

ax2.bar(accel_clean.index, accel_clean.values, color=colors_clean,
        width=1.5, linewidth=0)
ax2.axhline(0, color='#999999', linewidth=0.7)
ax2.set_ylabel('Acceleration (bps)')
ax2.set_xlabel('Date')

# legend patches
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor=COLORS['pc3'], label='Accel compression'),
    Patch(facecolor=COLORS['pos_outer'], label='Accel widening'),
    Patch(facecolor=COLORS['mean'], label='Decel / Transition'),
]
ax2.legend(handles=legend_els, loc='upper left')
ax1.set_xlim(fast.dropna().index[0], fast.index[-1])

plt.tight_layout()
plt.show()

# --- Print current signal ---
interpretations = {
    'accel_compress': 'Regional compression trend is gaining momentum. Expect continued bid for LatAm LC. Flow likely skewed to receivers / duration longs.',
    'accel_widen': 'Regional widening trend is accelerating. Selling pressure building across the bloc. Watch for stop-loss flows amplifying moves.',
    'decel': 'Momentum trend is losing pace. Conviction is fading; watch for reversal or consolidation before next directional move.',
    'transition': 'Short and medium-term signals are conflicting. Market is at an inflection point. Positioning may be caught offsides.',
    'unknown': 'Insufficient data to classify current momentum state.',
}

fast_val = float(fast.iloc[-1]) if not fast.empty else float('nan')
slow_val = float(slow.iloc[-1]) if not slow.empty else float('nan')
cur_state = state_series.iloc[-1] if not state_series.empty else 'unknown'
cur_date = fast.index[-1].date()

fast_dir = 'compression' if fast_val < 0 else 'widening'
slow_dir = 'compression' if slow_val < 0 else 'widening'

state_labels = {
    'accel_compress': 'Accelerating compression',
    'accel_widen': 'Accelerating widening',
    'decel': 'Decelerating (pace is slowing)',
    'transition': 'Transition (signals conflicting)',
    'unknown': 'Unknown',
}

print(f'=== Client Flow Anticipation | {MODE}: {SELECTED} | {cur_date} ===')
print()
print(f'PC1 5d momentum:  {fast_val:+.1f} bps ({fast_dir})')
print(f'PC1 20d momentum: {slow_val:+.1f} bps ({slow_dir})')
print(f'Momentum state:   {state_labels.get(cur_state, cur_state)}')
print()
print(f'Interpretation: {interpretations.get(cur_state, "")}' )
